<a href="https://colab.research.google.com/github/caseygudwien/StatClubCommodityModel/blob/main/CommodityModel.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import numpy as np
from sklearn.preprocessing import StandardScaler, LabelEncoder
from typing import Dict


class HierarchicalCommodityForecaster(nn.Module):
    """
    Advanced model for commodity trade forecasting with hierarchical structure
    Handles: Categories (e.g., '111') -> Subcategories (e.g., '1111') -> Items (e.g., '11111')

    This model uses separate embedding layers for each level of the commodity code
    hierarchy and concatenates them to form a rich representation.
    """

    def __init__(self,
                 seq_input_size: int = 12,  # Time series features
                 hidden_size: int = 256,
                 num_layers: int = 3,
                 dropout: float = 0.3,
                 # Dict mapping level (e.g., 'l1') to num_embeddings
                 embedding_vocab_sizes: Dict[str, int] = {},
                 # Dict mapping level (e.g., 'l1') to embedding_dim
                 embedding_dims: Dict[str, int] = {},
                 use_attention: bool = True):
        super().__init__()

        self.use_attention = use_attention

        # --- Hierarchical Commodity Embeddings ---
        self.embedding_layers = nn.ModuleDict()
        total_embedding_dim = 0
        for level, vocab_size in embedding_vocab_sizes.items():
            dim = embedding_dims.get(level, 0)
            if dim > 0:
                self.embedding_layers[level] = nn.Embedding(vocab_size, dim)
                total_embedding_dim += dim

        print(f"Total embedding dimension: {total_embedding_dim}")

        # Bidirectional LSTM for temporal patterns
        self.lstm = nn.LSTM(
            input_size=seq_input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            dropout=dropout if num_layers > 1 else 0,
            batch_first=True,
            bidirectional=True
        )

        lstm_output_size = hidden_size * 2  # Bidirectional

        # Multi-head attention for important time steps
        if use_attention:
            self.attention = nn.MultiheadAttention(
                embed_dim=lstm_output_size,
                num_heads=8,
                dropout=dropout,
                batch_first=True
            )

        # Seasonal decomposition branch
        self.seasonal_fc = nn.Sequential(
            nn.Linear(4, 32),  # month_sin, month_cos, year_trend, quarter
            nn.ReLU(),
            nn.Dropout(dropout)
        )

        # Combined features size
        combined_size = lstm_output_size + total_embedding_dim + 32

        # Separate heads for exports and imports
        self.fc_exports = nn.Sequential(
            nn.Linear(combined_size, 256),
            nn.LayerNorm(256),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Linear(64, 1)
        )

        self.fc_imports = nn.Sequential(
            nn.Linear(combined_size, 256),
            nn.LayerNorm(256),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Linear(64, 1)
        )

    def forward(self, seq, seasonal_features, commodity_ids: Dict[str, torch.Tensor]):
        """
        Args:
            seq: (batch, seq_len, features) - time series features
            seasonal_features: (batch, 4) - seasonal indicators
            commodity_ids: (batch,) - commodity code embeddings
        """
        # LSTM processing
        lstm_out, _ = self.lstm(seq)  # (batch, seq_len, hidden*2)

        # Apply attention if enabled
        if self.use_attention:
            attn_out, _ = self.attention(lstm_out, lstm_out, lstm_out)
            temporal_features = attn_out[:, -1, :]  # Last time step
        else:
            temporal_features = lstm_out[:, -1, :]

        # Commodity hierarchical embedding
        commodity_embeddings_list = []
        for level, embedding_layer in self.embedding_layers.items():
            commodity_embeddings_list.append(embedding_layer(commodity_ids[level]))

        commodity_emb = torch.cat(commodity_embeddings_list, dim=1)

        # Seasonal features
        seasonal_emb = self.seasonal_fc(seasonal_features)

        # Combine all features
        combined = torch.cat([temporal_features, commodity_emb, seasonal_emb], dim=1)

        # Predict exports and imports
        exports_pred = self.fc_exports(combined)
        imports_pred = self.fc_imports(combined)

        return torch.cat([exports_pred, imports_pred], dim=1)


class CommodityDataset(torch.utils.data.Dataset):
    """
    Dataset for hierarchical commodity data
    Handles seasonal patterns, hierarchical commodity codes, and log-scaling

    This version expects pre-fitted LabelEncoders for hierarchical codes.
    """

    def __init__(self, df,
                 commodity_encoders: Dict[str, LabelEncoder],
                 sequence_length=12,
                 scaler_X=None,
                 scaler_static=None,
                 log_offset=1.0,
                 clip_percentile=None):  # Added clip_percentile parameter
        self.sequence_length = sequence_length
        self.log_offset = log_offset
        self.commodity_encoders = commodity_encoders
        self.clip_percentile = clip_percentile # Store percentile

        df = df.copy()

        # --- Apply Hierarchical Encoders ---
        # Assume commodity codes are strings. Create hierarchical columns.
        # This parsing logic should match what was done to fit the encoders
        df['commodity_str'] = df['commodity'].astype(str)
        df['c_l1'] = df['commodity_str'].str.slice(0, 3) # e.g., '111'
        df['c_l2'] = df['commodity_str'].str.slice(0, 4) # e.g., '1111'
        df['c_l3'] = df['commodity_str'].str.slice(0, 5) # e.g., '11111'

        # Transform using the pre-fitted encoders
        for level in self.commodity_encoders.keys():
            df[f'c_{level}_enc'] = self.commodity_encoders[level].transform(df[f'c_{level}'])

        # --- Outlier Clipping ---
        if self.clip_percentile is not None:
            # Calculate clip values based on the training data if not provided
            if scaler_X is None: # Assuming scalers are only none for training data
                self.exports_upper_clip_value = df['exports'].quantile(self.clip_percentile / 100)
                self.imports_upper_clip_value = df['imports'].quantile(self.clip_percentile / 100)
                self.exports_lower_clip_value = df['exports'].quantile(1 - (self.clip_percentile / 100))
                self.imports_lower_clip_value = df['imports'].quantile(1 - (self.clip_percentile / 100))
                print(f"Clipping exports at upper: {self.exports_upper_clip_value:,.0f} (>{self.clip_percentile}th percentile), lower: {self.exports_lower_clip_value:,.0f} (<{100 - self.clip_percentile}th percentile)")
                print(f"Clipping imports at upper: {self.imports_upper_clip_value:,.0f} (>{self.clip_percentile}th percentile), lower: {self.imports_lower_clip_value:,.0f} (<{100 - self.clip_percentile}th percentile)")
            else:
                 # Use clip values from the training dataset
                 self.exports_upper_clip_value = scaler_X.exports_upper_clip_value # Assuming scaler_X stores clip values
                 self.imports_upper_clip_value = scaler_X.imports_upper_clip_value # Assuming scaler_X stores clip values
                 self.exports_lower_clip_value = scaler_X.exports_lower_clip_value
                 self.imports_lower_clip_value = scaler_X.imports_lower_clip_value


            # Apply clipping to both train, val, and test dataframes
            df['exports'] = df['exports'].clip(lower=self.exports_lower_clip_value, upper=self.exports_upper_clip_value)
            df['imports'] = df['imports'].clip(lower=self.imports_lower_clip_value, upper=self.imports_upper_clip_value)
            print(f"Applied clipping with {self.clip_percentile}th percentile threshold.")


        # Log transform trade values
        df['log_exports'] = np.log(df['exports'] + log_offset)
        df['log_imports'] = np.log(df['imports'] + log_offset)

        # Create seasonal features
        df['month_sin'] = np.sin(2 * np.pi * df['month'] / 12)
        df['month_cos'] = np.cos(2 * np.pi * df['month'] / 12)
        df['year_trend'] = (df['year'] - df['year'].min()) / (df['year'].max() - df['year'].min())
        df['quarter'] = ((df['month'] - 1) // 3) / 4  # Normalized quarter

        self.samples = []

        # Create sequences per commodity (using original commodity column for grouping)
        for commodity_code in df['commodity'].unique():
            commodity_data = df[df['commodity'] == commodity_code].sort_values(['year', 'month']).copy()

            if len(commodity_data) < sequence_length + 1:
                continue

            # Add momentum indicators
            commodity_data['exports_growth'] = commodity_data['exports'].pct_change().fillna(0)
            commodity_data['imports_growth'] = commodity_data['imports'].pct_change().fillna(0)
            commodity_data['log_exports_ma3'] = commodity_data['log_exports'].rolling(3, min_periods=1).mean()
            commodity_data['log_imports_ma3'] = commodity_data['log_imports'].rolling(3, min_periods=1).mean()
            commodity_data['volatility_exports'] = commodity_data['log_exports'].rolling(3, min_periods=1).std().fillna(0)
            commodity_data['volatility_imports'] = commodity_data['log_imports'].rolling(3, min_periods=1).std().fillna(0)

            commodity_data = commodity_data.reset_index(drop=True)

            for i in range(len(commodity_data) - sequence_length):
                seq = commodity_data.iloc[i:i + sequence_length]
                target_row = commodity_data.iloc[i + sequence_length]

                # Sequence features (12 features)
                seq_features = np.column_stack([
                    seq['log_exports'].values,
                    seq['log_imports'].values,
                    seq['month_sin'].values,
                    seq['month_cos'].values,
                    seq['year_trend'].values,
                    seq['quarter'].values,
                    seq['exports_growth'].values,
                    seq['imports_growth'].values,
                    seq['log_exports_ma3'].values,
                    seq['log_imports_ma3'].values,
                    seq['volatility_exports'].values,
                    seq['volatility_imports'].values,
                ])

                # Seasonal features for target period
                seasonal_features = np.array([
                    target_row['month_sin'],
                    target_row['month_cos'],
                    target_row['year_trend'],
                    target_row['quarter']
                ])

                # Hierarchical commodity IDs for target period
                commodity_id_dict = {}
                for level in self.commodity_encoders.keys():
                    commodity_id_dict[level] = target_row[f'c_{level}_enc']

                # Target (log-scaled)
                target = np.array([
                    target_row['log_exports'],
                    target_row['log_imports']
                ])

                self.samples.append((seq_features, seasonal_features, commodity_id_dict, target))

        print(f"Created {len(self.samples)} samples")

        # Fit scalers on *training* data (if not provided)
        if scaler_X is None:
            from sklearn.preprocessing import StandardScaler
            all_seq = np.vstack([s[0].reshape(-1, s[0].shape[-1]) for s in self.samples])
            self.scaler_X = StandardScaler()
            self.scaler_X.fit(all_seq)

            all_seasonal = np.vstack([s[1] for s in self.samples])
            self.scaler_static = StandardScaler()
            self.scaler_static.fit(all_seasonal)

            # Store clip values in scaler_X for use in val and test datasets
            if self.clip_percentile is not None:
                 self.scaler_X.exports_upper_clip_value = self.exports_upper_clip_value
                 self.scaler_X.imports_upper_clip_value = self.imports_upper_clip_value
                 self.scaler_X.exports_lower_clip_value = self.exports_lower_clip_value
                 self.scaler_X.imports_lower_clip_value = self.imports_lower_clip_value


        else:
            self.scaler_X = scaler_X
            self.scaler_static = scaler_static
            # Retrieve clip values from scaler_X if available
            if self.clip_percentile is not None:
                if hasattr(self.scaler_X, 'exports_upper_clip_value'):
                    self.exports_upper_clip_value = self.scaler_X.exports_upper_clip_value
                    self.imports_upper_clip_value = self.scaler_X.imports_upper_clip_value
                    self.exports_lower_clip_value = self.scaler_X.exports_lower_clip_value
                    self.imports_lower_clip_value = self.scaler_X.imports_lower_clip_value

                else:
                     print("Warning: Clip values not found in provided scaler_X. Clipping will not be applied consistently.")
                     self.clip_percentile = None # Disable clipping if values are not available


    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        seq, seasonal, commodity_id_dict, target = self.samples[idx]

        # Scale features
        seq_scaled = self.scaler_X.transform(seq.reshape(-1, seq.shape[-1])).reshape(seq.shape)
        seasonal_scaled = self.scaler_static.transform(seasonal.reshape(1, -1)).reshape(-1)

        # Convert commodity IDs to tensors
        commodity_ids_tensor_dict = {
            level: torch.LongTensor([id_val])[0]
            for level, id_val in commodity_id_dict.items()
        }

        return (
            torch.FloatTensor(seq_scaled),
            torch.FloatTensor(seasonal_scaled),
            commodity_ids_tensor_dict,
            torch.FloatTensor(target)
        )

    def inverse_transform_predictions(self, predictions):
        """Convert log predictions back to original scale"""
        return np.maximum(np.exp(predictions) - self.log_offset, 0)


def train_model(model: nn.Module,
                train_loader: DataLoader,
                val_loader: DataLoader,
                num_epochs: int = 100,
                learning_rate: float = 0.0005,
                export_weight: float = 1.0,
                import_weight: float = 0.8,
                device: str = 'cuda' if torch.cuda.is_available() else 'cpu'):
    """Train the forecasting model for commodity data"""

    model = model.to(device)
    criterion = nn.MSELoss()  # Use MSE for log-scale predictions
    optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate, weight_decay=0.01)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', patience=10, factor=0.5)

    best_val_loss = float('inf')
    train_losses = []
    val_losses = []

    for epoch in range(num_epochs):
        # Training
        model.train()
        train_loss = 0.0

        # UPDATED: Unpack 4 items, including commodity_id_dict
        for seq, seasonal, commodity_id_dict, target in train_loader:
            seq = seq.to(device)
            seasonal = seasonal.to(device)
            # Move commodity ID dict to device
            commodity_id_dict = {k: v.to(device) for k, v in commodity_id_dict.items()}
            target = target.to(device)

            optimizer.zero_grad()
            output = model(seq, seasonal, commodity_id_dict)  # Pass dict

            loss_exports = criterion(output[:, 0], target[:, 0])
            loss_imports = criterion(output[:, 1], target[:, 1])
            loss = export_weight * loss_exports + import_weight * loss_imports

            loss.backward()

            # Gradient clipping
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

            optimizer.step()
            train_loss += loss.item()

        train_loss /= len(train_loader)
        train_losses.append(train_loss)

        # Validation
        model.eval()
        val_loss = 0.0

        with torch.no_grad():
            for seq, seasonal, commodity_id_dict, target in val_loader:
                seq = seq.to(device)
                seasonal = seasonal.to(device)
                commodity_id_dict = {k: v.to(device) for k, v in commodity_id_dict.items()}
                target = target.to(device)

                output = model(seq, seasonal, commodity_id_dict)

                loss_exports = criterion(output[:, 0], target[:, 0])
                loss_imports = criterion(output[:, 1], target[:, 1])
                loss = export_weight * loss_exports + import_weight * loss_imports
                val_loss += loss.item()

        val_loss /= len(val_loader)
        val_losses.append(val_loss)

        # Learning rate scheduling
        scheduler.step(val_loss)

        # Save best model
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            torch.save(model.state_dict(), 'best_commodity_model_v3.pth')

        # Print progress every epoch
        print(f'Epoch [{epoch+1}/{num_epochs}], Train Loss: {train_loss:.4f}, Val Loss: {val_loss:.4f}')

    # Load best model
    model.load_state_dict(torch.load('best_commodity_model_v3.pth'))
    return model, train_losses, val_losses


def predict(model: nn.Module,
            data_loader: DataLoader,
            dataset: 'CommodityDataset',  # Pass the dataset for inverse transform
            device: str = 'cuda' if torch.cuda.is_available() else 'cpu'):
    """Make predictions and convert back from log scale"""

    model = model.to(device)
    model.eval()

    predictions = []
    actuals = []

    with torch.no_grad():
        # UPDATED: Unpack 4 items, including commodity_id_dict
        for seq, seasonal, commodity_id_dict, target in data_loader:
            seq = seq.to(device)
            seasonal = seasonal.to(device)
            commodity_id_dict = {k: v.to(device) for k, v in commodity_id_dict.items()}

            output = model(seq, seasonal, commodity_id_dict)
            predictions.append(output.cpu().numpy())
            actuals.append(target.numpy())

    predictions = np.vstack(predictions)
    actuals = np.vstack(actuals)

    # Convert from log scale back to original scale
    predictions_original = dataset.inverse_transform_predictions(predictions)
    actuals_original = dataset.inverse_transform_predictions(actuals)

    return predictions_original, actuals_original

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import torch
import torch.nn as nn
import numpy as np
import pandas as pd
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import LabelEncoder
# Import the updated classes and functions


def custom_collate_fn(batch):
    """Custom collate function to handle dictionary of commodity IDs"""
    seqs = torch.stack([item[0] for item in batch])
    seasonals = torch.stack([item[1] for item in batch])
    targets = torch.stack([item[3] for item in batch])

    # Handle commodity_id_dict - stack each level separately
    commodity_id_dicts = [item[2] for item in batch]
    levels = commodity_id_dicts[0].keys()

    batched_commodity_ids = {}
    for level in levels:
        batched_commodity_ids[level] = torch.stack([d[level] for d in commodity_id_dicts])

    return seqs, seasonals, batched_commodity_ids, targets


# --- This guard is the main fix for Windows multiprocessing ---
if __name__ == "__main__":

    # --- 1. Data Loading and Preprocessing ---
    print("Loading data...")
    df = pd.read_csv("/content/drive/MyDrive/Commodities Over Time(in).csv")
    df = df.rename(columns={
        'Commodity' : 'commodity',
        'Total Exports Value ($US)': 'exports',
        'Customs Import Value (Gen) ($US)': 'imports',
        'Year': 'year',
        'Month': 'month'
    })

    month_map = {
        'Jan': 1, 'Feb': 2, 'Mar': 3, 'Apr': 4, 'May': 5, 'Jun': 6,
        'Jul': 7, 'Aug': 8, 'Sep': 9, 'Oct': 10, 'Nov': 11, 'Dec': 12
    }
    df['month'] = df['month'].map(month_map)

    # Remove commas from numbers and convert to numeric
    df['exports'] = df['exports'].str.replace(',', '').astype(float)
    df['imports'] = df['imports'].str.replace(',', '').astype(float)
    df['year'] = pd.to_numeric(df['year'], errors='coerce')

    df = df.dropna(subset=['commodity', 'year', 'month', 'exports', 'imports'])
    df = df.reset_index(drop=True)
    df = df.sort_values(['commodity', 'year', 'month']).reset_index(drop=True)

    df['date'] = pd.to_datetime(df[['year', 'month']].assign(day=1))

    # --- 2. Create Hierarchical Encoders ---
    print("Creating hierarchical encoders...")
    df['commodity_str'] = df['commodity'].astype(str)
    df['c_l1'] = df['commodity_str'].str.slice(0, 3) # e.g., '111'
    df['c_l2'] = df['commodity_str'].str.slice(0, 4) # e.g., '1111'
    df['c_l3'] = df['commodity_str'].str.slice(0, 5) # e.g., '11111'

    commodity_encoders = {
        'l1': LabelEncoder().fit(df['c_l1']),
        'l2': LabelEncoder().fit(df['c_l2']),
        'l3': LabelEncoder().fit(df['c_l3'])
    }

    embedding_vocab_sizes = {
        level: len(encoder.classes_)
        for level, encoder in commodity_encoders.items()
    }
    print(f"Embedding vocab sizes: {embedding_vocab_sizes}")

    embedding_dims = {
        'l1': 16,
        'l2': 32,
        'l3': 64
    }


    # --- 3. Data Splitting (Time-based) ---
    print("Splitting data...")
    unique_dates = df['date'].unique()
    unique_dates = np.sort(unique_dates)

    train_cutoff_idx = int(len(unique_dates) * 0.7)
    val_cutoff_idx = int(len(unique_dates) * 0.85)

    train_cutoff = unique_dates[train_cutoff_idx]
    val_cutoff = unique_dates[val_cutoff_idx]

    train_df = df[df['date'] < train_cutoff].copy()
    val_df = df[(df['date'] >= train_cutoff) & (df['date'] < val_cutoff)].copy()
    test_df = df[df['date'] >= val_cutoff].copy()

    # --- 4. Create Datasets and DataLoaders ---
    sequence_length = 12

    print("Initializing train dataset (this may take a moment)...")
    train_dataset = CommodityDataset(train_df,
                                     commodity_encoders=commodity_encoders,
                                     sequence_length=sequence_length,
                                     clip_percentile=2) # Set clip_percentile to 2 for 2% clipping

    print("Initializing val dataset...")
    val_dataset = CommodityDataset(val_df,
                                   commodity_encoders=commodity_encoders,
                                   sequence_length=sequence_length,
                                   scaler_X=train_dataset.scaler_X,
                                   scaler_static=train_dataset.scaler_static,
                                   clip_percentile=2) # Set clip_percentile to 2 for 2% clipping

    print("Initializing test dataset...")
    test_dataset = CommodityDataset(test_df,
                                    commodity_encoders=commodity_encoders,
                                    sequence_length=sequence_length,
                                    scaler_X=train_dataset.scaler_X,
                                    scaler_static=train_dataset.scaler_static,
                                    clip_percentile=2) # Set clip_percentile to 2 for 2% clipping

    # --- FIX: Use custom collate function ---
    print("Creating DataLoaders...")
    device = 'cuda' if torch.cuda.is_available() else 'cpu'

    loader_args = {
        'batch_size': 32,
        'num_workers': 0,
        'pin_memory': False,
        'collate_fn': custom_collate_fn  # Add custom collate function
    }

    train_loader = DataLoader(train_dataset, shuffle=True, **loader_args)
    val_loader = DataLoader(val_dataset, shuffle=False, **loader_args)
    test_loader = DataLoader(test_dataset, shuffle=False, **loader_args)

    # --- 5. Model Initialization ---
    print("Initializing model...")
    model = HierarchicalCommodityForecaster(
        seq_input_size=12,
        hidden_size=256,
        num_layers=3,
        embedding_vocab_sizes=embedding_vocab_sizes,
        embedding_dims=embedding_dims,
        use_attention=True
    )

    # --- 6. Model Training ---
    print(f"Starting training on {device}...")
    model, train_losses, val_losses = train_model(
        model, train_loader, val_loader,
        num_epochs=100,
        learning_rate=0.0001,
        export_weight=1.0,
        import_weight=1.5,
        device=device
    )

    # --- 7. Prediction and Evaluation ---
    print("Training complete. Running predictions on test set...")
    predictions, actuals = predict(model, test_loader, test_dataset, device=device)

    threshold = 10000
    exports_actual = actuals[:, 0]
    exports_pred = predictions[:, 0]
    imports_actual = actuals[:, 1]
    imports_pred = predictions[:, 1]

    mask_exports = exports_actual > threshold
    mask_imports = imports_actual > threshold

    print(f"\n=== FILTERED EVALUATION (threshold > ${threshold:,}) ===")
    print(f"Exports samples used: {mask_exports.sum()} / {len(exports_actual)}")
    print(f"Imports samples used: {mask_imports.sum()} / {len(imports_actual)}")

    if mask_exports.sum() > 0:
        exports_mape = np.mean(np.abs((exports_actual[mask_exports] - exports_pred[mask_exports]) / exports_actual[mask_exports])) * 100
        exports_rmse = np.sqrt(np.mean((exports_actual[mask_exports] - exports_pred[mask_exports]) ** 2))
        print(f"\nExports MAPE: {exports_mape:.2f}%")
        print(f"Exports RMSE: ${exports_rmse:,.0f}")
        exports_medape = np.median(np.abs((exports_actual[mask_exports] - exports_pred[mask_exports]) / exports_actual[mask_exports])) * 100
        print(f"Exports MedAPE: {exports_medape:.2f}%")

    if mask_imports.sum() > 0:
        imports_mape = np.mean(np.abs((imports_actual[mask_imports] - imports_pred[mask_imports]) / imports_actual[mask_imports])) * 100
        imports_rmse = np.sqrt(np.mean((imports_actual[mask_imports] - imports_pred[mask_imports]) ** 2))
        print(f"\nImports MAPE: {imports_mape:.2f}%")
        print(f"Imports RMSE: ${imports_rmse:,.0f}")
        imports_medape = np.median(np.abs((imports_actual[mask_imports] - imports_pred[mask_imports]) / imports_actual[mask_imports])) * 100
        print(f"Imports MedAPE: {imports_medape:.2f}%")

    mask_both = (exports_actual > threshold) | (imports_actual > threshold)
    if mask_both.sum() > 0:
        all_actual = np.concatenate([exports_actual[mask_exports], imports_actual[mask_imports]])
        all_pred = np.concatenate([exports_pred[mask_exports], imports_pred[mask_imports]])

        overall_mape = np.mean(np.abs((all_actual - all_pred) / all_actual)) * 100
        overall_rmse = np.sqrt(np.mean((all_actual - all_pred) ** 2))

        print(f"\nOverall MAPE (filtered): {overall_mape:.2f}%")
        print(f"Overall RMSE (filtered): ${overall_rmse:,.0f}")

    print("\n=== Sample Predictions ===")
    print("Exports (Predicted | Actual | Error %):")
    for i in range(min(10, len(predictions))):
        if exports_actual[i] > threshold:
            error_pct = abs(exports_pred[i] - exports_actual[i]) / exports_actual[i] * 100
            print(f"  ${exports_pred[i]:>15,.0f} | ${exports_actual[i]:>15,.0f} | {error_pct:>6.1f}%")

    print("\nImports (Predicted | Actual | Error %):")
    for i in range(min(10, len(predictions))):
        if imports_actual[i] > threshold:
            error_pct = abs(imports_pred[i] - imports_actual[i]) / imports_actual[i] * 100
            print(f"  ${imports_pred[i]:>15,.0f} | ${imports_actual[i]:>15,.0f} | {error_pct:>6.1f}%")

    print("\nDone.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Loading data...
Creating hierarchical encoders...
Embedding vocab sizes: {'l1': 33, 'l2': 141, 'l3': 366}
Splitting data...
Initializing train dataset (this may take a moment)...
Clipping exports at upper: 513,378 (>2th percentile), lower: 5,476,295,170 (<98th percentile)
Clipping imports at upper: 524,873 (>2th percentile), lower: 9,655,282,671 (<98th percentile)
Applied clipping with 2th percentile threshold.
Created 46940 samples
Initializing val dataset...
Applied clipping with 2th percentile threshold.
Created 2938 samples
Initializing test dataset...
Applied clipping with 2th percentile threshold.
Created 3564 samples
Creating DataLoaders...
Initializing model...
Total embedding dimension: 112
Starting training on cuda...
Epoch [1/100], Train Loss: 63.9136, Val Loss: 0.8489
Epoch [2/100], Train Loss: 6.7004, Val Loss: 0.4778
Epoch [3/100], Train Loss: 5

In [ ]:
import pandas as pd

# Get unique commodity codes from the original dataframe
unique_commodities = df['commodity'].unique()

# Create a list to hold the data for the new dataframe
data = []

# Generate rows for each commodity and each month in 2024
for commodity in unique_commodities:
    for month in range(1, 13):
        data.append([commodity, 2024, month, '', '', '', '', '', '', '']) # Add empty strings for other columns

# Create the new dataframe
new_df = pd.DataFrame(data, columns=['Commodity', 'Year', 'Month', 'Total Exports Value ($US)', 'Customs Import Value (Gen) ($US)', 'Total Imports Value ($US)', 'Customs Export Value (Gen) ($US)', 'Total Exports Value ($US).1', 'Customs Import Value (Gen) ($US).1', 'Total Imports Value ($US).1'])

# Save the dataframe to a CSV file
new_df.to_csv('commodity_2024_template.csv', index=False)

print("Created 'commodity_2024_template.csv'")

Created 'commodity_2024_template.csv'


In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler, LabelEncoder
from typing import Dict
import warnings

# Suppress SettingWithCopyWarning
warnings.simplefilter(action='ignore', category=pd.errors.SettingWithCopyWarning)

# ==============================================================================
# 1. RE-DEFINE YOUR MODEL AND DATASET CLASSES (Copy from your training script)
# ==============================================================================

class HierarchicalCommodityForecaster(nn.Module):
    """
    Advanced model for commodity trade forecasting with hierarchical structure
    Handles: Categories (e.g., '111') -> Subcategories (e.g., '1111') -> Items (e.g., '11111')

    This model uses separate embedding layers for each level of the commodity code
    hierarchy and concatenates them to form a rich representation.
    """

    def __init__(self,
                 seq_input_size: int = 12,  # Time series features
                 hidden_size: int = 256,
                 num_layers: int = 3,
                 dropout: float = 0.3,
                 # Dict mapping level (e.g., 'l1') to num_embeddings
                 embedding_vocab_sizes: Dict[str, int] = {},
                 # Dict mapping level (e.g., 'l1') to embedding_dim
                 embedding_dims: Dict[str, int] = {},
                 use_attention: bool = True):
        super().__init__()

        self.use_attention = use_attention

        # --- Hierarchical Commodity Embeddings ---
        self.embedding_layers = nn.ModuleDict()
        total_embedding_dim = 0
        for level, vocab_size in embedding_vocab_sizes.items():
            dim = embedding_dims.get(level, 0)
            if dim > 0:
                self.embedding_layers[level] = nn.Embedding(vocab_size, dim)
                total_embedding_dim += dim

        # Bidirectional LSTM for temporal patterns
        self.lstm = nn.LSTM(
            input_size=seq_input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            dropout=dropout if num_layers > 1 else 0,
            batch_first=True,
            bidirectional=True
        )

        lstm_output_size = hidden_size * 2  # Bidirectional

        # Multi-head attention for important time steps
        if use_attention:
            self.attention = nn.MultiheadAttention(
                embed_dim=lstm_output_size,
                num_heads=8,
                dropout=dropout,
                batch_first=True
            )

        # Seasonal decomposition branch
        self.seasonal_fc = nn.Sequential(
            nn.Linear(4, 32),  # month_sin, month_cos, year_trend, quarter
            nn.ReLU(),
            nn.Dropout(dropout)
        )

        # Combined features size
        combined_size = lstm_output_size + total_embedding_dim + 32

        # Separate heads for exports and imports
        self.fc_exports = nn.Sequential(
            nn.Linear(combined_size, 256),
            nn.LayerNorm(256),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Linear(64, 1)
        )

        self.fc_imports = nn.Sequential(
            nn.Linear(combined_size, 256),
            nn.LayerNorm(256),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Linear(64, 1)
        )

    def forward(self, seq, seasonal_features, commodity_ids: Dict[str, torch.Tensor]):
        """
        Args:
            seq: (batch, seq_len, features) - time series features
            seasonal_features: (batch, 4) - seasonal indicators
            commodity_ids: (batch,) - commodity code embeddings
        """
        # LSTM processing
        lstm_out, _ = self.lstm(seq)  # (batch, seq_len, hidden*2)

        # Apply attention if enabled
        if self.use_attention:
            attn_out, _ = self.attention(lstm_out, lstm_out, lstm_out)
            temporal_features = attn_out[:, -1, :]  # Last time step
        else:
            temporal_features = lstm_out[:, -1, :]

        # Commodity hierarchical embedding
        commodity_embeddings_list = []
        for level, embedding_layer in self.embedding_layers.items():
            commodity_embeddings_list.append(embedding_layer(commodity_ids[level]))

        commodity_emb = torch.cat(commodity_embeddings_list, dim=1)

        # Seasonal features
        seasonal_emb = self.seasonal_fc(seasonal_features)

        # Combine all features
        combined = torch.cat([temporal_features, commodity_emb, seasonal_emb], dim=1)

        # Predict exports and imports
        exports_pred = self.fc_exports(combined)
        imports_pred = self.fc_imports(combined)

        return torch.cat([exports_pred, imports_pred], dim=1)


# ==============================================================================
# 2. SETUP AND LOAD ASSETS
# ==============================================================================

# --- Configuration (should match your training script) ---
MODEL_PATH = 'best_commodity_model_v3.pth'
DATA_PATH = '/content/drive/MyDrive/Commodities Over Time(in).csv'
SEQUENCE_LENGTH = 12
LOG_OFFSET = 1.0
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

# --- Load and preprocess the full historical dataset ---
print("Loading and preprocessing historical data...")
df = pd.read_csv(DATA_PATH)
df = df.rename(columns={
    'Commodity' : 'commodity',
    'Total Exports Value ($US)': 'exports',
    'Customs Import Value (Gen) ($US)': 'imports',
    'Year': 'year',
    'Month': 'month'
})
month_map = {
    'Jan': 1, 'Feb': 2, 'Mar': 3, 'Apr': 4, 'May': 5, 'Jun': 6,
    'Jul': 7, 'Aug': 8, 'Sep': 9, 'Oct': 10, 'Nov': 11, 'Dec': 12
}
df['month'] = df['month'].map(month_map)
df['exports'] = df['exports'].str.replace(',', '').astype(float)
df['imports'] = df['imports'].str.replace(',', '').astype(float)
df['year'] = pd.to_numeric(df['year'], errors='coerce')
df = df.dropna(subset=['commodity', 'year', 'month', 'exports', 'imports'])
df = df.sort_values(['commodity', 'year', 'month']).reset_index(drop=True)
df['date'] = pd.to_datetime(df[['year', 'month']].assign(day=1))

# --- Re-create the encoders and fit on the full dataset ---
print("Re-creating encoders...")
df['commodity_str'] = df['commodity'].astype(str)
df['c_l1'] = df['commodity_str'].str.slice(0, 3)
df['c_l2'] = df['commodity_str'].str.slice(0, 4)
df['c_l3'] = df['commodity_str'].str.slice(0, 5)

commodity_encoders = {
    'l1': LabelEncoder().fit(df['c_l1']),
    'l2': LabelEncoder().fit(df['c_l2']),
    'l3': LabelEncoder().fit(df['c_l3'])
}

embedding_vocab_sizes = {level: len(encoder.classes_) for level, encoder in commodity_encoders.items()}
embedding_dims = {'l1': 16, 'l2': 32, 'l3': 64}

# Re-create the scalers by fitting them on the training data portion
print("Re-creating scalers...")
train_cutoff = df['date'].quantile(0.7, interpolation='nearest')
train_df = df[df['date'] < train_cutoff]

# Create temporary dataset just to get the scalers
temp_train_dataset = CommodityDataset(train_df, commodity_encoders, sequence_length=SEQUENCE_LENGTH, clip_percentile=2)
scaler_X = temp_train_dataset.scaler_X
scaler_static = temp_train_dataset.scaler_static

#  Load the trained model
print("Loading the trained model...")
model = HierarchicalCommodityForecaster(
    seq_input_size=12,
    hidden_size=256,
    num_layers=3,
    embedding_vocab_sizes=embedding_vocab_sizes,
    embedding_dims=embedding_dims,
    use_attention=True
)
model.load_state_dict(torch.load(MODEL_PATH, map_location=torch.device(DEVICE)))
model.to(DEVICE)
model.eval()
print(f"Model loaded and set to '{DEVICE}' for evaluation.")


# AUTOREGRESSIVE FORECASTING

print("\nStarting 2024 forecast...")
all_predictions = []
unique_commodities = df['commodity'].unique()
num_commodities = len(unique_commodities)

for i, commodity_code in enumerate(unique_commodities):
    print(f"  > Forecasting for commodity {commodity_code} ({i+1}/{num_commodities})")

    # Get the last SEQUENCE_LENGTH months of data for this commodity
    commodity_history = df[df['commodity'] == commodity_code].tail(SEQUENCE_LENGTH).copy()

    if len(commodity_history) < SEQUENCE_LENGTH:
        print(f"    - Skipping commodity {commodity_code}: not enough historical data ({len(commodity_history)} months).")
        continue

    # Loop through the 12 months of 2024
    for month_num in range(1, 13):
        target_year = 2024
        target_month = month_num

        # Prepare inputs from the current history
        history_df = commodity_history

        # Log transform
        history_df['log_exports'] = np.log(history_df['exports'] + LOG_OFFSET)
        history_df['log_imports'] = np.log(history_df['imports'] + LOG_OFFSET)

        # Seasonal features
        history_df['month_sin'] = np.sin(2 * np.pi * history_df['month'] / 12)
        history_df['month_cos'] = np.cos(2 * np.pi * history_df['month'] / 12)
        # We use a fixed year range for trend consistency during prediction
        history_df['year_trend'] = (history_df['year'] - df['year'].min()) / (df['year'].max() - df['year'].min())
        history_df['quarter'] = ((history_df['month'] - 1) // 3) / 4

        # Momentum features
        history_df['exports_growth'] = history_df['exports'].pct_change().fillna(0)
        history_df['imports_growth'] = history_df['imports'].pct_change().fillna(0)
        history_df['log_exports_ma3'] = history_df['log_exports'].rolling(3, min_periods=1).mean()
        history_df['log_imports_ma3'] = history_df['log_imports'].rolling(3, min_periods=1).mean()
        history_df['volatility_exports'] = history_df['log_exports'].rolling(3, min_periods=1).std().fillna(0)
        history_df['volatility_imports'] = history_df['log_imports'].rolling(3, min_periods=1).std().fillna(0)

        # Create sequence features
        seq_features_np = history_df[[
            'log_exports', 'log_imports', 'month_sin', 'month_cos', 'year_trend',
            'quarter', 'exports_growth', 'imports_growth', 'log_exports_ma3',
            'log_imports_ma3', 'volatility_exports', 'volatility_imports'
        ]].values

        # Prepare seasonal features for the *target* month
        target_seasonal_features_np = np.array([
            np.sin(2 * np.pi * target_month / 12),
            np.cos(2 * np.pi * target_month / 12),
            (target_year - df['year'].min()) / (df['year'].max() - df['year'].min()),
            ((target_month - 1) // 3) / 4
        ])

        # Prepare hierarchical commodity IDs
        commodity_str = str(commodity_code)
        commodity_id_dict = {
            level: torch.LongTensor([encoder.transform([commodity_str[:len(encoder.classes_[0])]])[0]]).to(DEVICE)
            for level, encoder in commodity_encoders.items()
        }


        # Scale, convert to tensor, and add batch dimension
        seq_scaled = scaler_X.transform(seq_features_np)
        seasonal_scaled = scaler_static.transform(target_seasonal_features_np.reshape(1, -1)).flatten()

        seq_tensor = torch.FloatTensor(seq_scaled).unsqueeze(0).to(DEVICE)
        seasonal_tensor = torch.FloatTensor(seasonal_scaled).unsqueeze(0).to(DEVICE)

        # Make prediction
        with torch.no_grad():
            log_prediction = model(seq_tensor, seasonal_tensor, commodity_id_dict)

        # Inverse transform the prediction
        prediction_np = log_prediction.cpu().numpy()
        prediction_original = np.maximum(np.exp(prediction_np) - LOG_OFFSET, 0).flatten()
        predicted_exports = prediction_original[0]
        predicted_imports = prediction_original[1]

        # Store the result
        all_predictions.append({
            'commodity': commodity_code,
            'year': target_year,
            'month': target_month,
            'predicted_exports': predicted_exports,
            'predicted_imports': predicted_imports
        })

        #  Update history for the next iteration
        new_row = pd.DataFrame([{
            'commodity': commodity_code,
            'year': target_year,
            'month': target_month,
            'exports': predicted_exports,
            'imports': predicted_imports,
            'date': pd.to_datetime(f"{target_year}-{target_month}-01")
        }])
        commodity_history = pd.concat([commodity_history, new_row]).iloc[1:]


print("\nForecast complete. Saving results...")
forecast_df = pd.DataFrame(all_predictions)
forecast_df.to_csv('forecast_2024.csv', index=False)

print("Successfully saved 2024 forecast to 'forecast_2024.csv'")
print(forecast_df.head())

Loading and preprocessing historical data...
Re-creating encoders...
Re-creating scalers...
Clipping exports at upper: 513,378 (>2th percentile), lower: 5,476,295,170 (<98th percentile)
Clipping imports at upper: 524,873 (>2th percentile), lower: 9,655,282,671 (<98th percentile)
Applied clipping with 2th percentile threshold.
Created 46940 samples
Loading the trained model...
Model loaded and set to 'cuda' for evaluation.

Starting 2024 forecast...
  > Forecasting for commodity 111 Agricultural Products (1/805)
  > Forecasting for commodity 1111 Oilseeds & Grains (2/805)
  > Forecasting for commodity 11111 Soybeans (3/805)
  > Forecasting for commodity 111110 Soybeans (4/805)
  > Forecasting for commodity 11112 Oilseeds (except Soybean) (5/805)
  > Forecasting for commodity 111120 Oilseeds (except Soybean) (6/805)
  > Forecasting for commodity 11113 Dry Peas & Beans (7/805)
  > Forecasting for commodity 111130 Dry Peas & Beans (8/805)
  > Forecasting for commodity 11114 Wheat (9/805)
 